# 06 — Minimal repro: vLLM `draft_model` cannot load compressed-tensors checkpoints

**Purpose**: a small, self-contained, publicly-runnable reproduction of the bug
found in `docs/findings.md` 2026-07-24 (`ValueError: ... weight_packed ...`) while
trying to use our own SGT-QAT Qwen3-1.7B checkpoint as a vLLM speculative-decoding
drafter for Qwen3-8B. That original repro used a private checkpoint (Google Drive)
and a mixed-precision (W4/W3) recipe — neither is necessary to trigger the bug, and
neither is runnable by anyone outside this project. **This notebook isolates the
actual bug**: any `save_compressed=True` compressed-tensors checkpoint, loaded via
`speculative_config={"method": "draft_model", ...}`, hits the same failure —
regardless of model size or whether the recipe is mixed-precision.

Uses a tiny public model (`Qwen/Qwen3-0.6B`) as both target and draft, quantized
with a single plain GPTQ `oneshot()` call (W4A16, no mixed precision, no QAT) —
deliberately as cheap and minimal as possible so this is fast to verify before
filing, and so any vLLM maintainer can run it in a couple of minutes.

**Not yet run.** Run this before filing the issue in `docs/vllm-bug-report-draft.md`
— confirms the minimal repro actually reproduces the same failure on a fresh,
small checkpoint, rather than assuming it does.

## 1. Environment info for the bug report

Run this first — `docs/vllm-bug-report-draft.md` has placeholders for both outputs
below. Paste them back in exactly as printed, don't summarize/truncate.

In [ ]:
!pip show vllm
print("\n" + "="*80 + "\n")
!python -m vllm.collect_env

## 2. Build a tiny compressed-tensors checkpoint (plain W4A16 GPTQ, no mixed precision)

In [ ]:
!pip install -q llmcompressor compressed-tensors

import torch
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

MODEL_ID = 'Qwen/Qwen3-0.6B'  # small on purpose -- this repro doesn't need our 1.7B/8B pair
SEQ_LEN = 2048
CALIB_N = 32  # small on purpose -- this is a loading-path bug, not a quality benchmark
CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4a16-compressed-repro')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)

ds = load_dataset('allenai/c4', 'en', split='train', streaming=True).shuffle(seed=42, buffer_size=10_000)
samples, collected = [], 0
for item in ds:
    enc = tokenizer(item['text'], return_tensors='pt', truncation=True, max_length=SEQ_LEN)
    if enc['input_ids'].shape[1] == SEQ_LEN:
        samples.append(enc['input_ids'])
        collected += 1
        if collected >= CALIB_N:
            break
calib_dataset = Dataset.from_dict({'input_ids': torch.cat(samples, dim=0).tolist()})

# Plain W4A16, single scheme, no config_groups/mixed precision -- the simplest
# possible thing that still produces a genuinely packed compressed-tensors checkpoint.
recipe = GPTQModifier(targets='Linear', ignore=['lm_head'], scheme='W4A16', dampening_frac=0.01)
oneshot(model=model, dataset=calib_dataset, recipe=recipe, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)

model.save_pretrained(str(CHECKPOINT_DIR), save_compressed=True)
tokenizer.save_pretrained(str(CHECKPOINT_DIR))

# Sanity check: confirm this genuinely saved packed weights, not a full-precision
# fallback -- look for weight_packed in the safetensors keys before even trying
# to load it into vLLM.
from safetensors import safe_open
shard = next(CHECKPOINT_DIR.glob('*.safetensors'))
with safe_open(str(shard), framework='pt') as f:
    keys = list(f.keys())
has_packed = any('weight_packed' in k for k in keys)
print(f"Checkpoint saved to {CHECKPOINT_DIR}, contains weight_packed tensors: {has_packed}")
assert has_packed, "Checkpoint didn't actually save as compressed -- nothing to repro here."

del model
torch.cuda.empty_cache()

## 3. Trigger the bug: load it as a vLLM `draft_model`

Expected (per the bug): this raises `ValueError: There is no module or parameter
named '...weight_packed' in ... . The available parameters belonging to ... are:
{'...weight'}` — the same shape of error as our original repro, just against a
different (tiny, public, unmixed-precision) checkpoint. If it does NOT raise this
error, the bug may be fixed in this vLLM version, or something about the minimal
repro doesn't match our original setup closely enough -- either way, don't paste
the original (unverified-on-this-version) traceback into the issue; use whatever
this cell actually produces.

In [ ]:
import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'  # so the real traceback surfaces here, not in a swallowed subprocess
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

from vllm import LLM

llm = LLM(
    model=MODEL_ID,  # same tiny model as target, for simplicity -- the bug is in draft-model loading, not target/draft compatibility
    speculative_config={
        'method': 'draft_model',
        'model': str(CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

## 4. Copy the exact traceback from the cell above into `docs/vllm-bug-report-draft.md`

Paste the full traceback (not just the last line) into the "Actual behavior"
section, and the outputs from cell 1 into "Your current environment".